In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# RTE Datasets
The RTE (Recognizing Textual Entailment) datasets were re-annotated for contradiction by

> de Marneffe, M.-C., Rafferty, A. N., & Manning, C. D. (2008). Finding contradictions in text. In *Proceedings of ACL-08: HLT* (pp. 1039–1047). Association for Computational Linguistics. https://aclanthology.org/P08-1118/

The underlying text-hypothesis pairs originate from the PASCAL Recognizing Textual Entailment Challenges, which were organized to evaluate systems' ability to determine whether a given text (the "hypothesis") can be inferred from another text (the "text"). The datasets consist of pairs of sentences labeled as "entailment", "contradiction", or "neutral", depending on the relationship between the two sentences.

> Giampiccolo, D., Magnini, B., Dagan, I., & Dolan, B. (2007). The third PASCAL recognizing textual entailment challenge. In *Proceedings of the ACL-PASCAL Workshop on Textual Entailment and Paraphrasing* (pp. 1–9). Association for Computational Linguistics. https://aclanthology.org/W07-1401/

> Dagan, I., Glickman, O., & Magnini, B. (2006). The PASCAL recognising textual entailment challenge. In J. Quiñonero-Candela, I. Dagan, B. Magnini, & F. d'Alché-Buc (Eds.), *Machine learning challenges: Evaluating predictive uncertainty, visual object classification, and recognising textual entailment* (Lecture Notes in Computer Science, Vol. 3944, pp. 177–190). Springer. https://doi.org/10.1007/11736790_9

---

There are tasks associated with the RTE datasets, including:
```python
task_mapping = {
    "PP": "Paraphrase Acquisition",     # RTE-1 only (not on ACL Anthology)
    "RC": "Reading Comprehension",      # RTE-1 only (not on ACL Anthology)
    "IR": "Information Retrieval",      # confirmed: Giampiccolo et al. 2007 §1.1
    "QA": "Question Answering",         # confirmed: Giampiccolo et al. 2007 §1.1
    "CD": "Comparable Documents",       # RTE-1 only (not on ACL Anthology)
    "IE": "Information Extraction",     # confirmed: Giampiccolo et al. 2007 §1.1
    "MT": "Machine Translation",        # confirmed: Giampiccolo et al. 2007 §1.1
    "SUM": "Summarization",             # confirmed: Giampiccolo et al. 2007 §2.1 (RTE-2 onward)
}
```

**These task labels are not used in the `Finding Contradictions in Text` paper, so we can ignore them.**

What we are going to use are the entailment labels, contradiction type labels, and the sentence pairs.

The `real_contradiction` corpus is **not** part of RTE 1/2/3. It is a separate collection of 131 naturally occurring contradictions gathered by de Marneffe et al. (2008, §3.3) from Wikipedia (51), LDC data for the DARPA GALE distillation task (51), newswire via Google News (19), and the Lexis Nexis database (10). It is the only subset that carries contradiction type labels (antonym, negation, numeric, factive, structure, lexical, WK).

In [65]:
def pretty_print_rte(df: pd.DataFrame, negated: bool = False):
    if negated:
        print("Contradiction Distribution:")
        print(df["contradiction"].value_counts())
    else:
        print("Entailment Distribution:")
        print(df["entailment"].value_counts())
    print("─" * 100)  # Alt + 2500 (Box Drawings Light Horizontal)
    for i, row in enumerate(df.head(5).itertuples(index=False), start=1):
        if negated:
            print(f"Example {i} | ID: {row.id} | Contradiction: {row.contradiction}")
        else:
            print(f"Example {i} | ID: {row.id} | Entailment: {row.entailment}")
        print(f"Text:       {row.t}")
        print(f"Hypothesis: {row.h}")
        print("─" * 100)  # Alt + 2500 (Box Drawings Light Horizontal)

In [40]:
rte1_dev1 = pd.read_xml("data/raw/finding_contradictions_in_text_2008/RTE1_dev1_3ways.xml")
rte1_dev2 = pd.read_xml("data/raw/finding_contradictions_in_text_2008/RTE1_dev2_3ways.xml")
rte1_test = pd.read_xml("data/raw/finding_contradictions_in_text_2008/RTE1_test_3ways.xml")
rte2_dev = pd.read_xml("data/raw/finding_contradictions_in_text_2008/RTE2_dev_3ways.xml")
rte2_test = pd.read_xml("data/raw/finding_contradictions_in_text_2008/RTE2_test_3ways.xml")
rte2_dev_negated = pd.read_xml("data/raw/finding_contradictions_in_text_2008/RTE2_dev_negated_contradiction.xml")
rte2_test_negated = pd.read_xml("data/raw/finding_contradictions_in_text_2008/RTE2_test_negated_contradiction.xml")
rte3_dev = pd.read_xml("data/raw/finding_contradictions_in_text_2008/RTE3_dev_3ways.xml")
rte3_test = pd.read_xml("data/raw/finding_contradictions_in_text_2008/RTE3_test_3ways.xml")
real_contradiction = pd.read_xml("data/raw/finding_contradictions_in_text_2008/real_contradiction.xml")

In [41]:
rte1_dev1

,id,entailment,task,t,h
0,8,NO,IR,Crude oil for April delivery traded at $37.80 ...,Crude oil prices rose to $37.80 per barrel
1,12,NO,IR,Oracle had fought to keep the forms from being...,Oracle released a confidential document
2,13,YES,IR,iTunes software has seen strong sales in Europe.,Strong sales for iTunes in Europe.
3,15,NO,IR,"All genetically modified food, including soya ...",Companies selling genetically modified foods d...
4,19,YES,IR,Researchers at the Harvard School of Public He...,Coffee drinking has health benefits.
...,...,...,...,...,...
282,972,UNKNOWN,PP,The news comes as the owners of Kilroot announ...,Owners announced plans to sell clean-up equipm...
283,269,UNKNOWN,IR,U.S. Embassy spokesman Stanley Shrager said th...,Americans hope to help Evans Paul in the elect...
284,270,UNKNOWN,IR,"Like the United States, U.N. officials are als...",Aristide had Prime Minister Robert Malval mur...
285,275,UNKNOWN,IR,"Gladys Lauture, a close Aristide friend and lo...",Aristide is due to live at Port-au-Prince again.


In [66]:
pretty_print_rte(rte1_dev1)

Entailment Distribution:
entailment
YES        142
UNKNOWN     97
NO          48
Name: count, dtype: int64
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 1 | ID: 8 | Entailment: NO
Text:       Crude oil for April delivery traded at $37.80 a barrel, down 28 cents
Hypothesis: Crude oil prices rose to $37.80 per barrel
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 2 | ID: 12 | Entailment: NO
Text:       Oracle had fought to keep the forms from being released
Hypothesis: Oracle released a confidential document
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 3 | ID: 13 | Entailment: YES
Text:       iTunes software has seen strong sales in Europe.
Hypothesis: Strong sales for iTunes in Europe.
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 4 | ID: 

In [47]:
rte1_dev2

,id,entailment,task,t,h
0,2161,UNKNOWN,CD,City officials fired the captain of the crashe...,Staten Island ferry captain has refused to tal...
1,702,YES,CD,Budapest again became the focus of national po...,In the late 1980s Budapest became the center o...
2,2166,UNKNOWN,CD,"In Wednesday's filing, Google said it planned ...",It's unclear whether Wednesday's twist will af...
3,2174,UNKNOWN,CD,Al-Jazeera broadcasts to millions of Arab view...,Al-Jazeera has occasionally run into problems ...
4,818,YES,CD,The new study refutes earlier findings by rese...,The latest findings contradict a California st...
...,...,...,...,...,...
275,908,UNKNOWN,RC,Time Warner is the world's largest media and I...,Time Warner is the world's largest company.
276,975,YES,RC,After watching fireworks yesterday evening in ...,Kerry watched fireworks.
277,912,YES,RC,Vice President Dick Cheney on Tuesday hurled a...,Cheney cursed at Sen. Patrick Leahy.
278,331,UNKNOWN,RC,Chancellor Schroeder has presided over three y...,More than four million people have remained st...


In [67]:
pretty_print_rte(rte1_dev2)

Entailment Distribution:
entailment
YES        140
UNKNOWN     85
NO          55
Name: count, dtype: int64
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 1 | ID: 2161 | Entailment: UNKNOWN
Text:       City officials fired the captain of the crashed Staten Island ferry.
Hypothesis: Staten Island ferry captain has refused to talk with investigators.
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 2 | ID: 702 | Entailment: YES
Text:       Budapest again became the focus of national political drama in the late 1980s, when Hungary led the reform movement in eastern Europe that broke the communist monopoly on political power and ushered in the possibility of multiparty politics.
Hypothesis: In the late 1980s Budapest became the center of the reform movement.
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 3 | I

In [50]:
rte1_test

,id,entailment,task,t,h
0,754,YES,CD,Mexico City has a very bad pollution problem b...,Poor air circulation out of the mountain-walle...
1,822,YES,CD,Satomi Mitarai died of blood loss.,Satomi Mitarai bled to death.
2,692,YES,CD,"The national insurrection of 1794, led by Tade...",Warsaw became a town in Prussia.
3,731,UNKNOWN,CD,The city Tenochtitlan grew rapidly and was the...,"Tenochtitlan quickly spread over the island, m..."
4,1865,UNKNOWN,CD,Militant groups in Iraq have waged a kidnappin...,About 70 foreigners have been abducted by mili...
...,...,...,...,...,...
795,1062,YES,RC,A former petty thief who converted and founded...,Silva used to be a petty thief.
796,997,UNKNOWN,RC,One of Michigan's largest newspaper chains agr...,"The Journal Register Co. of Trenton, N.J. is M..."
797,1835,NO,RC,The new arrivals followed a similar number tha...,Thousands have fled South Korea's famine and r...
798,2124,UNKNOWN,RC,Fighters loyal to Moqtada al-Sadr shot down a ...,Fighters loyal to Moqtada al-Sadr shot down Na...


In [68]:
pretty_print_rte(rte1_test)

Entailment Distribution:
entailment
YES        400
UNKNOWN    251
NO         149
Name: count, dtype: int64
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 1 | ID: 754 | Entailment: YES
Text:       Mexico City has a very bad pollution problem because the mountains around the city act as walls and block in dust and smog.
Hypothesis: Poor air circulation out of the mountain-walled Mexico City aggravates pollution.
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 2 | ID: 822 | Entailment: YES
Text:       Satomi Mitarai died of blood loss.
Hypothesis: Satomi Mitarai bled to death.
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 3 | ID: 692 | Entailment: YES
Text:       The national insurrection of 1794, led by Tadeusz Kosciuszko against the Russo-Prussian second partition, was brutally crushed; the ensuing third

In [52]:
rte2_dev

,id,entailment,task,t,h
0,3,NO,IE,"ECB spokeswoman, Regina Schueller, declined to...",Regina Shueller works for Italy's La Repubblic...
1,4,UNKNOWN,IE,"Meanwhile, in an exclusive interview with a TI...","Ahmedinejad was attacked by the US, France, Br..."
2,11,NO,IE,The chaotic situation unleashed in Bogota last...,Justice Carlos Valencia was killed on 28 July.
3,12,UNKNOWN,IE,"He met U.S. President, George W. Bush, in Wash...",Washington is part of London.
4,13,YES,IE,Sunday's earthquake was felt in the southern I...,The city of Madras is located in Southern India.
...,...,...,...,...,...
795,789,UNKNOWN,SUM,Rumsfeld signaled a harder line against China ...,Rumsfeld is leveling some harsh criticism at C...
796,794,YES,SUM,Witching hour passed and Potter fans poured in...,Potter fans rushed to tills in order to purcha...
797,798,UNKNOWN,SUM,Rising house and stock prices have made many p...,A long spell of low interest rates and low ris...
798,799,YES,SUM,"Chinese Premier, Zhu Rongji, and German Chance...",Both Mr. Schroeder and Mr. Zhu have much to ga...


In [69]:
pretty_print_rte(rte2_dev)

Entailment Distribution:
entailment
YES        400
UNKNOWN    289
NO         111
Name: count, dtype: int64
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 1 | ID: 3 | Entailment: NO
Text:       ECB spokeswoman, Regina Schueller, declined to comment on a report in Italy's La Repubblica newspaper that the ECB council will discuss Mr. Fazio's role in the takeover fight at its Sept. 15 meeting.
Hypothesis: Regina Shueller works for Italy's La Repubblica newspaper.
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 2 | ID: 4 | Entailment: UNKNOWN
Text:       Meanwhile, in an exclusive interview with a TIME journalist, the first one-on-one session given to a Western print publication since his election as president of Iran earlier this year, Ahmadinejad attacked the "threat" to bring the issue of Iran's nuclear activity to the UN Security Council by the US, France, Britain a

In [54]:
rte2_test

,id,entailment,task,t,h
0,8,UNKNOWN,IE,Mangla was summoned after Madhumita's sister N...,Shukla is related to Mangla.
1,9,NO,IE,Authorities in Brazil say that more than 200 p...,Authorities in Brazil hold 200 people as hostage.
2,15,YES,IE,A mercenary group faithful to the warmongering...,An interior ministry worker was killed by a me...
3,16,YES,IE,"The British ambassador to Egypt, Derek Plumbly...",Derek Plumbly resides in Egypt.
4,17,YES,IE,Tibone estimated diamond production at four mi...,Botswana is a business partner of De Beers.
...,...,...,...,...,...
795,788,YES,SUM,Former Prime Minister Mahmoud Abbas declared v...,Mahmoud Abbas won the vote in the Palestinian ...
796,789,UNKNOWN,SUM,Packard Co. will start selling its version of ...,Packard Co. Friday unveiled its version of the...
797,790,YES,SUM,The BBC has apologised for incorrectly broadca...,The BBC reported that coalition and Iraqi forc...
798,794,UNKNOWN,SUM,The Israeli army said in a statement that the ...,The IDF said the air force targeted a car carr...


In [70]:
pretty_print_rte(rte2_test)

Entailment Distribution:
entailment
YES        400
UNKNOWN    296
NO         104
Name: count, dtype: int64
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 1 | ID: 8 | Entailment: UNKNOWN
Text:       Mangla was summoned after Madhumita's sister Nidhi Shukla, who was the first witness in the case.
Hypothesis: Shukla is related to Mangla.
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 2 | ID: 9 | Entailment: NO
Text:       Authorities in Brazil say that more than 200 people are being held hostage in a prison in the country's remote, Amazonian-jungle state of Rondonia.
Hypothesis: Authorities in Brazil hold 200 people as hostage.
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 3 | ID: 15 | Entailment: YES
Text:       A mercenary group faithful to the warmongering policy of former Somozist colonel Enrique Berm

In [56]:
rte3_dev

,id,entailment,task,length,t,h
0,1,YES,IE,short,The sale was made to pay Yukos' US$ 27.5 billi...,Baikalfinansgroup was sold to Rosneft.
1,2,NO,IE,short,The sale was made to pay Yukos' US$ 27.5 billi...,Yuganskneftegaz cost US$ 27.5 billion.
2,3,UNKNOWN,IE,long,Loraine besides participating in Broadway's Dr...,"""Does A Tiger Have A Necktie"" was produced in ..."
3,4,YES,IE,long,"""The Extra Girl"" (1923) is a story of a small-...","""The Extra Girl"" was produced by Sennett."
4,5,YES,IE,short,A bus collision with a truck in Uganda has res...,30 die in a bus collision in Uganda.
...,...,...,...,...,...,...
795,796,UNKNOWN,SUM,short,Confident that those grading papers would unde...,Haque wants to include English in some exams.
796,797,YES,SUM,short,"Iscor, the South African iron and steel manufa...",The South African steel manufacturer Iscor pla...
797,798,UNKNOWN,SUM,short,Critics said the National Certificate of Educa...,The NCEA has been degraded by the authority.
798,799,NO,SUM,short,"Two other marines, Tyler Jackson and John Jodk...",Tyler Jackson has been sentenced to 18 months.


In [71]:
pretty_print_rte(rte3_dev)

Entailment Distribution:
entailment
YES        412
UNKNOWN    308
NO          80
Name: count, dtype: int64
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 1 | ID: 1 | Entailment: YES
Text:       The sale was made to pay Yukos' US$ 27.5 billion tax bill, Yuganskneftegaz was originally sold for US$ 9.4 billion to a little known company Baikalfinansgroup which was later bought by the Russian state-owned oil company Rosneft .
Hypothesis: Baikalfinansgroup was sold to Rosneft.
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 2 | ID: 2 | Entailment: NO
Text:       The sale was made to pay Yukos' US$ 27.5 billion tax bill, Yuganskneftegaz was originally sold for US$9.4 billion to a little known company Baikalfinansgroup which was later bought by the Russian state-owned oil company Rosneft .
Hypothesis: Yuganskneftegaz cost US$ 27.5 billion.
─────────────────────────────────

In [58]:
rte3_test

,id,entailment,task,length,t,h
0,1,YES,IE,short,"Claude Chabrol (born June 24, 1930) is a Frenc...",Le Beau Serge was directed by Chabrol.
1,2,YES,IE,short,"Claude Chabrol (born June 24, 1930) is a Frenc...",Le Boucher was made by a French movie director.
2,3,YES,IE,short,"Claude Chabrol divorced Agnes, his first wife,...",Aurore Paquiss married Chabrol.
3,4,YES,IE,short,The Communist Party USA was a small Maoist pol...,Michael Laski was a member of a Maoist politic...
4,5,NO,IE,short,The Communist Party USA was a small Maoist pol...,Michael Laski was an opponent of China.
...,...,...,...,...,...,...
795,796,YES,SUM,short,It has pledged to allow the yuan to float more...,China has an export-led economy.
796,797,NO,SUM,short,Last year the US saw exports of goods and serv...,US exports rose 10.5%.
797,798,UNKNOWN,SUM,short,"The world's largest consumer of oil, US import...",There was a conflict between Israel and Lebanon.
798,799,UNKNOWN,SUM,short,"Away from the Olympic site, near the bricks an...",Aliens have been seen near the Forbidden City.


In [72]:
pretty_print_rte(rte3_test)

Entailment Distribution:
entailment
YES        410
UNKNOWN    318
NO          72
Name: count, dtype: int64
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 1 | ID: 1 | Entailment: YES
Text:       Claude Chabrol (born June 24, 1930) is a French movie director and has become well-known in the 40 years since his first film, Le Beau Serge , for his chilling tales of murder, including Le Boucher .
Hypothesis: Le Beau Serge was directed by Chabrol.
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 2 | ID: 2 | Entailment: YES
Text:       Claude Chabrol (born June 24, 1930) is a French movie director and has become well-known in the 40 years since his first film, Le Beau Serge , for his chilling tales of murder, including Le Boucher .
Hypothesis: Le Boucher was made by a French movie director.
────────────────────────────────────────────────────────────────────────────────────

## RTE-2 Negated Contradiction Datasets

The `rte2_dev_negated` and `rte2_test_negated` datasets are **not** from the original RTE-2 challenge. They are simulations of the LCC negation corpus (Harabagiu et al., 2006), built by adding negation markers to RTE-2 pairs (de Marneffe et al., 2008, §3.3). Because the task is contradiction detection rather than entailment, the label column is `contradiction` (binary: YES/NO), not `entailment` (3-way: YES/NO/UNKNOWN).

### Labeling rule

Let H' be the hypothesis with a negation marker inserted.

| Original RTE-2 label | Transformed pair | New `contradiction` label | Why |
|---|---|---|---|
| `entailment = YES` | (T, H') | `YES` | If T entails H, then T is inconsistent with ¬H. |
| `entailment = NO`  | (T, H') | `NO`  | T didn't entail H, and negations are hand-filtered so T and ¬H stay compatible. |

**Example (YES case)**: T = "Bill finished his math", H = "Bill has finished his math". Negating H gives H' = "Bill hasn't finished his math". T asserts he finished, H' asserts he didn't: contradiction.

**Example (NO case)**: T and H are already on different topics, so negating H yields an H' that is still non-contradictory with T.

### Why negation also appears in the NO rows

> "negative markers were also added to each non-entailment, ensuring that they did not create contradictions." (§3.3)

If only the YES rows contained "not", a classifier could cheat by learning "contains negation → contradiction". Spreading negation across both classes forces the model to judge whether the negation actually conflicts with T, rather than spotting the word.

### Logical justification

Entailment is monotonic over negation: if T ⊨ H then T ∧ ¬H is unsatisfiable, which is why flipping the YES rows into contradictions is sound. The NO rows are *not* guaranteed to avoid contradiction by logic alone (sometimes T ∧ ¬H clashes via world knowledge), so the annotators hand-filtered those to keep the `NO` label honest.

### Row counts

- `rte2_dev_negated`: ~100 pairs (50 entailments + 50 non-entailments sampled from RTE2-dev)
- `rte2_test_negated`: ~800 pairs (full RTE2-test with negation added to every pair)
- Both are balanced binary sets, which is why Table 5 of the paper reports **accuracy** for them (75.49% Neg_dev, 62.74% Neg_test) but not for the 3-way RTE sets.

In [60]:
rte2_dev_negated

,id,task,contradiction,t,h
0,4,IE,NO,"Meanwhile, in an exclusive interview with a TI...","Ahmedinejad was never attacked by the US, Fran..."
1,87,IE,YES,"Mexico City (Mexico), 12 Jan 90 (DPA) - Salvad...",Hector Oqueli Colindres does not come from El ...
2,90,IE,NO,"From Joss Stone to Keith Richards to Sting, th...",Eric Clapton is not in business with Les Paul.
3,166,IE,YES,"Although Schneider's father, Marvin, died of c...",Pilar is not the mother of Schneider.
4,171,IE,NO,Hinostroza's children told police that four ho...,Hinostroza's children never attaked their moth...
...,...,...,...,...,...
97,686,QA,NO,"Immediately after 1990, however, Swedish ferti...",Sweden is not the European country with the hi...
98,687,QA,YES,"As with the Babylonian New Year, the Chinese N...",The Chinese New Year's Day does not fall on th...
99,743,QA,NO,Mr. Balasingham will return to his London home...,Vilupillai Prabhakaran is not a diplomat.
100,761,QA,NO,Apart from the new substances entering the mar...,New substances are not damaging the ozone layer.


In [73]:
pretty_print_rte(rte2_dev_negated, negated=True)

Contradiction Distribution:
contradiction
NO     51
YES    51
Name: count, dtype: int64
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 1 | ID: 4 | Contradiction: NO
Text:       Meanwhile, in an exclusive interview with a TIME journalist, the first one-on-one session given to a Western print publication since his election as president of Iran earlier this year, Ahmadinejad attacked the "threat" to bring the issue of Iran's nuclear activity to the UN Security Council by the US, France, Britain and Germany.
Hypothesis: Ahmedinejad was never attacked by the US, France, Britain and Germany.
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 2 | ID: 87 | Contradiction: YES
Text:       Mexico City (Mexico), 12 Jan 90 (DPA) - Salvadoran Social Democratic politician Hector Oqueli Colindres was kidnapped today, in Guatemala city, along with Guatemalan Social Democratic leader G

In [21]:
rte2_test_negated

,id,contradiction,t,h
0,15,YES,A mercenary group faithful to the warmongering...,No interior ministry worker was killed by a me...
1,16,YES,"The British ambassador to Egypt, Derek Plumbly...",Derek Plumbly does not reside in Egypt.
2,17,YES,Tibone estimated diamond production at four mi...,Botswana is not a business partner of De Beers.
3,23,YES,His wife Strida won a seat in parliament after...,Strida was not elected to parliament.
4,27,YES,Responding to Scheuer's comments in La Repubbl...,Mel Sembler does not represent the U.S.
...,...,...,...,...
792,773,NO,The militant group led by Abu Musab al-Zarqawi...,Abu Musab al-Zarqawi's group will not behead o...
793,787,NO,Commandos stormed a school Friday in southern ...,Hundreds of people were not killed after terro...
794,789,NO,Packard Co. will start selling its version of ...,Packard Co. Friday did not unveil its version ...
795,794,NO,The Israeli army said in a statement that the ...,The IDF never said the air force targeted a ca...


In [75]:
pretty_print_rte(rte2_test_negated, negated=True)

Contradiction Distribution:
contradiction
YES    400
NO     397
Name: count, dtype: int64
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 1 | ID: 15 | Contradiction: YES
Text:       A mercenary group faithful to the warmongering policy of former Somozist colonel Enrique Bermudez attacked an IFA truck belonging to the interior ministry at 0900 on 26 March in El Jicote, wounded and killed an interior ministry worker and wounded five others.
Hypothesis: No interior ministry worker was killed by a mercenary group.
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 2 | ID: 16 | Contradiction: YES
Text:       The British ambassador to Egypt, Derek Plumbly, told Reuters on Monday that authorities had compiled the list of 10 based on lists from tour companies and from families whose relatives have not been in contact since the bombings.
Hypothesis: Derek Plumbly does not resid

# This is the main dataset that we will use for our contradiction detection experiments.

In [ ]:
real_contradiction

,id,contradiction,type,t,h
0,1,YES,negation,Tariq Aziz was not considered a member of Sadd...,Tariq Aziz was in Saddam's inner circle.
1,2,YES,lexical,Tariq Aziz kept outside the closed circle of S...,Tariq Aziz was in Saddam's inner circle.
2,3,YES,negation,Tariq Aziz was not one of the most powerful fi...,Tariq Aziz was prominent in the regime.
3,4,YES,lexical,Tariq Aziz retained influence.,Aziz had virtually no power.
4,5,YES,numeric,Alexandros Yiotopoulos is 58.,Alexandros Yiotopoulos is 59 years old.
...,...,...,...,...,...
126,127,YES,numeric,It has been criticized as pseudoscientific qua...,It has been criticized as pseudoscientific qua...
127,128,YES,numeric,He became a Field Marshal in the British Army ...,He became a Field Marshal in the British Army ...
128,129,YES,structure,"Currently, I-75 is 15 lanes wide at the Windy ...",Bisecting the city from west to east across it...
129,130,YES,numeric,"In Turkey, there are around 820 higher educati...",The total capacity of Turkish universities is ...


In [82]:
def pretty_print_real_contradiction(df: pd.DataFrame):
    print("Contradiction Distribution:")
    print(df["contradiction"].value_counts())
    print("─" * 100)  # Alt + 2500 (Box Drawings Light Horizontal)
    print(df["type"].value_counts())
    print("─" * 100)  # Alt + 2500 (Box Drawings Light Horizontal)
    for i, row in enumerate(df.head(5).itertuples(index=False), start=1):
        print(f"Example {i} | ID: {row.id} | Contradiction: {row.contradiction} | Type: {row.type}")
        print(f"Text:       {row.t}")
        print(f"Hypothesis: {row.h}")
        print("─" * 100)  # Alt + 2500 (Box Drawings Light Horizontal)

In [83]:
pretty_print_real_contradiction(real_contradiction)

Contradiction Distribution:
contradiction
YES    131
Name: count, dtype: int64
────────────────────────────────────────────────────────────────────────────────────────────────────
type
numeric        38
lexical        28
negation       23
WK             13
antonym        12
factive         9
structure       4
WK_temporal     3
WK_location     1
Name: count, dtype: int64
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 1 | ID: 1 | Contradiction: YES | Type: negation
Text:       Tariq Aziz was not considered a member of Saddam's innermost circle.
Hypothesis: Tariq Aziz was in Saddam's inner circle.
────────────────────────────────────────────────────────────────────────────────────────────────────
Example 2 | ID: 2 | Contradiction: YES | Type: lexical
Text:       Tariq Aziz kept outside the closed circle of Saddam's Sunni Moslem cronies.
Hypothesis: Tariq Aziz was in Saddam's inner circle.
───────────────────────────────────────

# Combime datasets based on types

In [98]:
entailment_3ways_df_train = pd.concat([rte1_dev1, rte1_dev2, rte2_dev, rte3_dev], ignore_index=True).drop(columns=["id", "task", "length"])
entailment_3ways_df_train["entailment"].value_counts()

entailment
YES        1094
UNKNOWN     779
NO          294
Name: count, dtype: int64

In [99]:
entailment_3ways_df_train

,entailment,t,h
0,NO,Crude oil for April delivery traded at $37.80 ...,Crude oil prices rose to $37.80 per barrel
1,NO,Oracle had fought to keep the forms from being...,Oracle released a confidential document
2,YES,iTunes software has seen strong sales in Europe.,Strong sales for iTunes in Europe.
3,NO,"All genetically modified food, including soya ...",Companies selling genetically modified foods d...
4,YES,Researchers at the Harvard School of Public He...,Coffee drinking has health benefits.
...,...,...,...
2162,UNKNOWN,Confident that those grading papers would unde...,Haque wants to include English in some exams.
2163,YES,"Iscor, the South African iron and steel manufa...",The South African steel manufacturer Iscor pla...
2164,UNKNOWN,Critics said the National Certificate of Educa...,The NCEA has been degraded by the authority.
2165,NO,"Two other marines, Tyler Jackson and John Jodk...",Tyler Jackson has been sentenced to 18 months.


In [100]:
entailment_3ways_df_test = pd.concat([rte1_test, rte2_test, rte3_test], ignore_index=True).drop(columns=["id", "task", "length"])
entailment_3ways_df_test["entailment"].value_counts()

entailment
YES        1210
UNKNOWN     865
NO          325
Name: count, dtype: int64

In [101]:
entailment_3ways_df_test

,entailment,t,h
0,YES,Mexico City has a very bad pollution problem b...,Poor air circulation out of the mountain-walle...
1,YES,Satomi Mitarai died of blood loss.,Satomi Mitarai bled to death.
2,YES,"The national insurrection of 1794, led by Tade...",Warsaw became a town in Prussia.
3,UNKNOWN,The city Tenochtitlan grew rapidly and was the...,"Tenochtitlan quickly spread over the island, m..."
4,UNKNOWN,Militant groups in Iraq have waged a kidnappin...,About 70 foreigners have been abducted by mili...
...,...,...,...
2395,YES,It has pledged to allow the yuan to float more...,China has an export-led economy.
2396,NO,Last year the US saw exports of goods and serv...,US exports rose 10.5%.
2397,UNKNOWN,"The world's largest consumer of oil, US import...",There was a conflict between Israel and Lebanon.
2398,UNKNOWN,"Away from the Olympic site, near the bricks an...",Aliens have been seen near the Forbidden City.


In [106]:
# Since original train only have around 100 but test have around 800, combine them and split later to have more balanced distribution
contradiction_binary_df = pd.concat([rte2_dev_negated, rte2_test_negated], ignore_index=True).drop(columns=["id", "task"])
contradiction_binary_df["contradiction"].value_counts()

contradiction
YES    451
NO     448
Name: count, dtype: int64

In [107]:
contradiction_binary_df

,contradiction,t,h
0,NO,"Meanwhile, in an exclusive interview with a TI...","Ahmedinejad was never attacked by the US, Fran..."
1,YES,"Mexico City (Mexico), 12 Jan 90 (DPA) - Salvad...",Hector Oqueli Colindres does not come from El ...
2,NO,"From Joss Stone to Keith Richards to Sting, th...",Eric Clapton is not in business with Les Paul.
3,YES,"Although Schneider's father, Marvin, died of c...",Pilar is not the mother of Schneider.
4,NO,Hinostroza's children told police that four ho...,Hinostroza's children never attaked their moth...
...,...,...,...
894,NO,The militant group led by Abu Musab al-Zarqawi...,Abu Musab al-Zarqawi's group will not behead o...
895,NO,Commandos stormed a school Friday in southern ...,Hundreds of people were not killed after terro...
896,NO,Packard Co. will start selling its version of ...,Packard Co. Friday did not unveil its version ...
897,NO,The Israeli army said in a statement that the ...,The IDF never said the air force targeted a ca...


In [109]:
real_contradiction_processed = real_contradiction.drop(columns=["id"])
real_contradiction_processed

,contradiction,type,t,h
0,YES,negation,Tariq Aziz was not considered a member of Sadd...,Tariq Aziz was in Saddam's inner circle.
1,YES,lexical,Tariq Aziz kept outside the closed circle of S...,Tariq Aziz was in Saddam's inner circle.
2,YES,negation,Tariq Aziz was not one of the most powerful fi...,Tariq Aziz was prominent in the regime.
3,YES,lexical,Tariq Aziz retained influence.,Aziz had virtually no power.
4,YES,numeric,Alexandros Yiotopoulos is 58.,Alexandros Yiotopoulos is 59 years old.
...,...,...,...,...
126,YES,numeric,It has been criticized as pseudoscientific qua...,It has been criticized as pseudoscientific qua...
127,YES,numeric,He became a Field Marshal in the British Army ...,He became a Field Marshal in the British Army ...
128,YES,structure,"Currently, I-75 is 15 lanes wide at the Windy ...",Bisecting the city from west to east across it...
129,YES,numeric,"In Turkey, there are around 820 higher educati...",The total capacity of Turkish universities is ...


In [110]:
output_df_path = "data/processed/finding_contradictions_in_text_2008/"
entailment_3ways_df_train.to_csv(output_df_path + "entailment_3ways_train.csv", index=False)
entailment_3ways_df_test.to_csv(output_df_path + "entailment_3ways_test.csv", index=False)
contradiction_binary_df.to_csv(output_df_path + "contradiction_binary.csv", index=False)
real_contradiction_processed.to_csv(output_df_path + "real_contradiction.csv", index=False)